# Part 3: Sepsis triage as a web service on IBM LinuxONE
The trained models are served as a REST API. Hospital or clinic systems send readings and get back the risk score, tier, alert and next step.

In [2]:
# Export autoencoder so we reduce dependencies
import json, numpy as np, torch
sd_ = torch.load('ae.pt')
np.savez('ae_encoder.npz',
         W1=sd_['encoder.0.weight'].numpy(), b1=sd_['encoder.0.bias'].numpy(),
         W2=sd_['encoder.3.weight'].numpy(), b2=sd_['encoder.3.bias'].numpy())

# Check if the NumPy version gives the same embedding as the PyTorch one
meta = json.load(open('model_meta.json'))
n = len(meta['feature_cols'])
enc = torch.nn.Sequential(torch.nn.Linear(n, 64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
                          torch.nn.Linear(64, meta['EMB_DIM']), torch.nn.ReLU())
enc.load_state_dict({k.replace('encoder.', ''): v for k, v in sd_.items() if k.startswith('encoder.')})
enc.eval()
x = np.random.randn(5, n).astype(np.float32)
E = np.load('ae_encoder.npz')
np_out = np.maximum(np.maximum(x @ E['W1'].T + E['b1'], 0) @ E['W2'].T + E['b2'], 0)
with torch.no_grad():
    t_out = enc(torch.tensor(x)).numpy()
print("Max difference NumPy vs PyTorch:", float(np.abs(np_out - t_out).max()))

Max difference NumPy vs PyTorch: 0.0


In [3]:
%%writefile app.py
# Sepsis triage API - runs on IBM LinuxONE (s390x). Hospital/clinic systems POST readings, get risk + next step.
import json, platform, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from flask import Flask, request, jsonify

with open('model_meta.json') as f:
    meta = json.load(f)
FEAT, LABS, VITALS = meta['feature_cols'], meta['lab_cols'], meta['vital_cols']
EMB, T1COLS = meta['emb_cols'], meta['tier1_cols']
MU = np.array([meta['mu'][c] for c in FEAT])
SD = np.array([meta['sd'][c] for c in FEAT])
TIER1 = lgb.Booster(model_file='tier1_lgb.txt')
TIER2 = lgb.Booster(model_file='tier2_lgb.txt')
ENC = np.load('ae_encoder.npz')          # autoencoder encoder weights (no PyTorch needed at serving time)

def embed(x):
    z = np.nan_to_num((x - MU) / SD, nan=0.0)
    h = np.maximum(z @ ENC['W1'].T + ENC['b1'], 0)
    return np.maximum(h @ ENC['W2'].T + ENC['b2'], 0)

CHECKS = [('HR',    lambda v: v > 90,           'heart rate {:.0f} bpm (above 90)'),
          ('Resp',  lambda v: v >= 22,          'breathing rate {:.0f}/min (22 or more)'),
          ('Temp',  lambda v: v > 38 or v < 36, 'temperature {:.1f} C (outside 36-38)'),
          ('SBP',   lambda v: v <= 100,         'systolic BP {:.0f} mmHg (100 or less)'),
          ('MAP',   lambda v: v < 65,           'mean arterial pressure {:.0f} mmHg (below 65)'),
          ('O2Sat', lambda v: v < 92,           'oxygen saturation {:.0f}% (below 92)')]

def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

app = Flask(__name__)

@app.get('/health')
def health():
    return jsonify(status='ok', platform=platform.machine(), models=['tier1_hospital', 'tier2_clinic'])

@app.post('/triage')
def triage():
    t0 = time.perf_counter()
    data = request.get_json(force=True) or {}
    r = {c: to_float(data.get(c)) for c in FEAT}
    if np.isnan(r['MAP']) and not np.isnan(r['SBP']) and not np.isnan(r['DBP']):
        r['MAP'] = (r['SBP'] + 2 * r['DBP']) / 3          # calculated, never guessed
    row = pd.DataFrame([r], columns=FEAT, dtype=float)

    if row[LABS].notna().any(axis=1).iloc[0]:
        feats = pd.concat([row, pd.DataFrame(embed(row.values), columns=EMB)], axis=1)
        tier, score, thr = 'tier1_hospital', float(TIER1.predict(feats[T1COLS])[0]), meta['thr1']
    else:
        tier, score, thr = 'tier2_clinic', float(TIER2.predict(row[VITALS])[0]), meta['thr2']
    alert = score >= thr
    flags = [t.format(r[c]) for c, bad, t in CHECKS if not np.isnan(r[c]) and bad(r[c])]
    if alert:
        step = 'Sepsis alert - request urgent review by a doctor.'
    elif flags:
        step = 'No alert - continue routine monitoring and re-check vitals within the hour.'
    else:
        step = 'No alert - continue routine monitoring.'
    return jsonify(tier=tier, risk_score=round(score, 4), alert_threshold=round(thr, 4), alert=bool(alert),
                   abnormal_vitals=flags, next_step=step, platform=platform.machine(),
                   latency_ms=round((time.perf_counter() - t0) * 1000, 2))

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8050)


Writing app.py


In [4]:
%%writefile requirements.txt
flask
numpy
pandas
lightgbm


Writing requirements.txt


In [5]:
%%writefile Containerfile
# Build & run on IBM LinuxONE (s390x) with Podman:
#   podman build -t sepsis-triage-api .
#   podman run -d -p 8050:8050 --name sepsis-api sepsis-triage-api
#   curl localhost:8050/health
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends cmake g++ make libgomp1 \
    && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py model_meta.json tier1_lgb.txt tier2_lgb.txt ae_encoder.npz ./
EXPOSE 8050
USER 1001
CMD ["python", "app.py"]


Writing Containerfile


In [6]:
import subprocess, sys, time, requests, platform
server = subprocess.Popen([sys.executable, 'app.py'], stdout=open('api.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Server is up:", requests.get('http://localhost:8050/health', timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start - see api.log:"); print(open('api.log').read())

Server is up: {'models': ['tier1_hospital', 'tier2_clinic'], 'platform': 's390x', 'status': 'ok'}


In [7]:
import pandas as pd, numpy as np, json
demo = pd.read_csv('demo_patients.csv')
p = demo.iloc[1]
payload = {c: (None if pd.isna(p[c]) else float(p[c])) for c in meta['feature_cols']}
print(f"Sending unseen test patient {int(p['Patient_ID'])}, hour {int(p['Hour'])} "
      f"(actual: {'SEPSIS' if p['SepsisLabel'] == 1 else 'no sepsis'})")
r = requests.post('http://localhost:8050/triage', json=payload)
print(json.dumps(r.json(), indent=2))

Sending unseen test patient 117517, hour 42 (actual: SEPSIS)
{
  "abnormal_vitals": [
    "heart rate 118 bpm (above 90)",
    "temperature 38.9 C (outside 36-38)",
    "systolic BP 100 mmHg (100 or less)"
  ],
  "alert": true,
  "alert_threshold": 0.5075,
  "latency_ms": 5.03,
  "next_step": "Sepsis alert - request urgent review by a doctor.",
  "platform": "s390x",
  "risk_score": 0.6162,
  "tier": "tier1_hospital"
}


In [8]:
!curl -s -X POST http://localhost:8050/triage -H "Content-Type: application/json" \
  -d '{"Age": 38, "Gender": 1, "HR": 118, "Resp": 26, "Temp": 38.9, "SBP": 92, "DBP": 55, "O2Sat": 92}'

{"abnormal_vitals":["heart rate 118 bpm (above 90)","breathing rate 26/min (22 or more)","temperature 38.9 C (outside 36-38)","systolic BP 92 mmHg (100 or less)"],"alert":true,"alert_threshold":0.5153,"latency_ms":2.25,"next_step":"Sepsis alert - request urgent review by a doctor.","platform":"s390x","risk_score":0.8597,"tier":"tier2_clinic"}


In [9]:
times = []
for i in range(200):
    p = demo.iloc[i % len(demo)]
    payload = {c: (None if pd.isna(p[c]) else float(p[c])) for c in meta['feature_cols']}
    t0 = time.perf_counter()
    requests.post('http://localhost:8050/triage', json=payload)
    times.append((time.perf_counter() - t0) * 1000)
print(f"200 API calls on {platform.machine()}: median {np.median(times):.1f} ms, "
      f"95th percentile {np.percentile(times, 95):.1f} ms per request (including network round trip)")

200 API calls on s390x: median 3.1 ms, 95th percentile 7.0 ms per request (including network round trip)


In [10]:
server.terminate()
print("Server stopped")

Server stopped
